# Build Constructor Standings

#### Sources
1. fact_session_results
1. dim_constructors

#### Output Columns
1. season
1. constructor id
1. constructor name
1. nationality
1. race starts
1. total points
1. number of wins
1. number of podiums
1. standing position


#### Entity Relationship Diagram - Formula1 Gold Schema

![Formula1 Gold Data.png](../z-course-images/formula1-gold-data-erd.png "Formula1 Gold Data.png")

In [0]:
CREATE OR REPLACE VIEW f1.gold.v_constructor_standing
AS
WITH constructor_session_summary
AS
  (SELECT r.season,
        c.constructor_id,
        c.constructor_name,
        c.nationality,
        COUNT(*) AS race_starts,
        SUM(r.points) AS total_points,
        COUNT_IF(r.is_win) AS number_of_wins,
        COUNT_IF(r.is_podium) AS number_of_podiums
    FROM f1.gold.fact_session_results r
    JOIN f1.gold.dim_constructors c
      ON r.constructor_id = c.constructor_id 
  GROUP BY r.season,
        c.constructor_id,
        c.constructor_name,
        c.nationality)    
SELECT season,
       constructor_id,
       constructor_name,
       nationality,
       RANK() OVER (PARTITION BY season ORDER BY total_points DESC, number_of_wins DESC) AS standing,
       race_starts,
       total_points,
       number_of_wins,
       number_of_podiums
  FROM constructor_session_summary;


In [0]:
SELECT * FROM f1.gold.v_constructor_standing WHERE season = 2025